# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ishpree1t7/flyrank_work/blob/main/work/notebooks/w06_validation_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding 1 — AI-search sessions increased substantially

The FlyRank research report describes a tracked portfolio where AI sessions increased from roughly 422 to 6.6K between October 2025 and March 2026.

Methodology question: How was an “AI session” defined and attributed in the underlying analytics data? Were AI referrals identifiable consistently across the whole period, and was the measurement coverage comparable at both endpoints? A large measured increase is useful, but the comparison is stronger if the measurement definition and coverage remained stable.

Finding 2 — Updated older content can approach newer content on content-health measures

The report compares an updated roughly one-year-old article with newer content and reports health scores of about 36.98 versus 39.02.

Methodology question: How was the health score constructed, and does the comparison control for factors other than age/update status, such as search demand, topic, authority, or content type? If this is an observational comparison, I would describe it as an observed association rather than evidence that updating older content caused the improvement.

These are constructive methodology questions rather than a grade of the research. The same questions should be applied to my own model: where does the label come from, what information was available at prediction time, and does the validation design support the strength of the claim?

In [2]:
import os

print("Current directory:")
print(os.getcwd())

print("\nFiles/folders here:")
print(os.listdir())

print("\nSearching for CSV files...")

csv_files = []

for root, dirs, files in os.walk("/content"):
    for file in files:
        if file.endswith(".csv"):
            csv_files.append(os.path.join(root, file))

print("\nCSV files found:")
for f in csv_files:
    print(f)

print("\nTotal CSV files:", len(csv_files))

Current directory:
/content

Files/folders here:
['.config', 'sample_data']

Searching for CSV files...

CSV files found:
/content/sample_data/california_housing_test.csv
/content/sample_data/mnist_test.csv
/content/sample_data/california_housing_train.csv
/content/sample_data/mnist_train_small.csv

Total CSV files: 4


In [3]:
!git clone https://github.com/ishpree1t7/flyrank_work.git

Cloning into 'flyrank_work'...
remote: Enumerating objects: 144, done.
remote: Counting objects: 100% (144/144), done.
remote: Compressing objects: 100% (99/99), done.
remote: Total 144 (delta 54), reused 99 (delta 29), pack-reused 0 (from 0)
Receiving objects: 100% (144/144), 1.88 MiB | 5.71 MiB/s, done.
Resolving deltas: 100% (54/54), done.


In [4]:
import os

print("Repo contents:")
print(os.listdir("/content/flyrank_work"))

print("\nSearching repo for CSV files...")

csv_files = []

for root, dirs, files in os.walk("/content/flyrank_work"):
    for file in files:
        if file.endswith(".csv"):
            csv_files.append(os.path.join(root, file))

for f in csv_files:
    print(f)

print("\nTotal CSV files:", len(csv_files))

Repo contents:
['DATA_USE.md', 'GUIDE.md', 'work', 'requirements.txt', 'outputs', 'submission', 'notebooks', '.gitignore', '.github', 'docs', '.git', 'AGENTS.md', 'CLAUDE.md', 'LICENSE', 'skills', 'README.md', 'SETUP.md', 'data', 'scripts']

Searching repo for CSV files...
/content/flyrank_work/outputs/refresh_queue_sample.csv
/content/flyrank_work/data/raw/content_refresh_anonymized.csv

Total CSV files: 2


In [5]:
import pandas as pd

DATA_PATH = "/content/flyrank_work/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset path:", DATA_PATH)
print("Shape:", df.shape)
print("Number of columns:", len(df.columns))

# Create the Week-5 target
df["is_declining_label"] = (
    df["trend_direction"].str.lower() == "down"
).astype(int)

print("\nDeclining rate:", round(df["is_declining_label"].mean(), 3))
print("Number of clients:", df["client_id"].nunique())

Dataset path: /content/flyrank_work/data/raw/content_refresh_anonymized.csv
Shape: (30000, 44)
Number of columns: 44

Declining rate: 0.542
Number of clients: 32


In [6]:
FEATURES = [
    "days_since_last_update",
    "impressions_90d"
]

TARGET = "is_declining_label"
GROUP = "client_id"

data = df[FEATURES + [TARGET, GROUP]].dropna().copy()

X = data[FEATURES]
y = data[TARGET]
groups = data[GROUP]

print("Rows used:", len(data))
print("Features:", FEATURES)
print("Target:", TARGET)
print("Unique clients:", groups.nunique())

Rows used: 30000
Features: ['days_since_last_update', 'impressions_90d']
Target: is_declining_label
Unique clients: 32


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

## My model under an honest split

The Week-5 model used a stratified random 80/20 split. This is useful as a baseline, but repeated rows from the same client can appear in both training and testing.

For this audit, I keep the same Week-5 features and logistic-regression approach, but use a **grouped split by `client_id`**.

This means the model is evaluated on clients that were not present in the training set.

The Week-5 random-split results were:

* Precision@10: **0.400**
* Precision@20: **0.250**
* Precision@50: **0.420**

I will compare these measurements with the grouped-by-client results below.


In [7]:
def precision_at_k(y_true, scores, k):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    k = min(k, len(y_true))

    top_indices = np.argsort(-scores)[:k]

    return float(y_true[top_indices].mean())


def make_model():
    return Pipeline([
        ("scale", StandardScaler()),
        ("model", LogisticRegression(
            random_state=42,
            max_iter=1000
        ))
    ])

In [8]:
# BEFORE: reproduce the Week-5 random split

X_train_random, X_test_random, y_train_random, y_test_random = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

random_model = make_model()

random_model.fit(
    X_train_random,
    y_train_random
)

random_scores = random_model.predict_proba(
    X_test_random
)[:, 1]

random_results = {
    "Precision@10": precision_at_k(
        y_test_random.reset_index(drop=True),
        random_scores,
        10
    ),
    "Precision@20": precision_at_k(
        y_test_random.reset_index(drop=True),
        random_scores,
        20
    ),
    "Precision@50": precision_at_k(
        y_test_random.reset_index(drop=True),
        random_scores,
        50
    )
}

print("BEFORE — Random split")
for metric, value in random_results.items():
    print(metric, ":", round(value, 3))

BEFORE — Random split
Precision@10 : 0.4
Precision@20 : 0.25
Precision@50 : 0.42


In [9]:
# AFTER: grouped split by client_id

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train_grouped = X.iloc[train_idx]
X_test_grouped = X.iloc[test_idx]

y_train_grouped = y.iloc[train_idx]
y_test_grouped = y.iloc[test_idx]

groups_train = groups.iloc[train_idx]
groups_test = groups.iloc[test_idx]

grouped_model = make_model()

grouped_model.fit(
    X_train_grouped,
    y_train_grouped
)

grouped_scores = grouped_model.predict_proba(
    X_test_grouped
)[:, 1]

grouped_results = {
    "Precision@10": precision_at_k(
        y_test_grouped.reset_index(drop=True),
        grouped_scores,
        10
    ),
    "Precision@20": precision_at_k(
        y_test_grouped.reset_index(drop=True),
        grouped_scores,
        20
    ),
    "Precision@50": precision_at_k(
        y_test_grouped.reset_index(drop=True),
        grouped_scores,
        50
    )
}

print("AFTER — Grouped by client")
for metric, value in grouped_results.items():
    print(metric, ":", round(value, 3))

print("\nTraining clients:", groups_train.nunique())
print("Testing clients:", groups_test.nunique())

overlap = set(groups_train) & set(groups_test)

print("Client overlap:", len(overlap))

AFTER — Grouped by client
Precision@10 : 0.4
Precision@20 : 0.5
Precision@50 : 0.5

Training clients: 25
Testing clients: 7
Client overlap: 0


In [10]:
comparison = pd.DataFrame({
    "Week-5 random split": random_results,
    "Grouped-by-client split": grouped_results
})

comparison["Change"] = (
    comparison["Grouped-by-client split"]
    - comparison["Week-5 random split"]
)

display(comparison.round(3))

,Week-5 random split,Grouped-by-client split,Change
Precision@10,0.40,0.4,0.00
Precision@20,0.25,0.5,0.25
Precision@50,0.42,0.5,0.08


### Before/after interpretation

The random split provides the Week-5 baseline, while the grouped split tests performance on clients that were not represented in training.

The difference between the two measurements shows how sensitive the observed ranking performance is to the validation design.

A lower grouped result would indicate that the random split may have benefited from client-specific structure shared between training and testing. A similar result would provide evidence that the measured signal is less dependent on seeing the same clients.

This experiment measures ranking performance under different validation designs. It does **not** establish that the model causes improved content performance.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

## Leakage audit

The Week-5 model uses only `days_since_last_update` and `impressions_90d`.

The target `is_declining_label` is derived from `trend_direction`, so `trend_direction` and other target-derived fields must not be used as model features.

`client_id` is also not used as a feature. It is used only to create the grouped validation split.

I will explicitly check that the final feature list does not contain the target, trend-derived fields, or identifiers.


In [11]:
forbidden_columns = {
    "is_declining_label",
    "trend_direction",
    "trend_pct",
    "client_id",
    "content_id"
}

print("Final features:")
for feature in FEATURES:
    print("-", feature)

leakage_found = set(FEATURES) & forbidden_columns

print("\nForbidden features found:", leakage_found)

if len(leakage_found) == 0:
    print("PASS: No obvious target-derived or ID leakage in final features.")
else:
    print("WARNING: Potential leakage detected.")

Final features:
- days_since_last_update
- impressions_90d

Forbidden features found: set()
PASS: No obvious target-derived or ID leakage in final features.


In [12]:
error_examples = X_test_grouped.copy()

error_examples["actual"] = y_test_grouped.to_numpy()
error_examples["model_score"] = grouped_scores
error_examples["client_id"] = groups_test.to_numpy()

# Highest-scored predictions
top_predictions = error_examples.sort_values(
    "model_score",
    ascending=False
).head(10)

print("Top 10 predictions on unseen clients:")
display(top_predictions)

Top 10 predictions on unseen clients:


,days_since_last_update,impressions_90d,actual,model_score,client_id
24152,124,4,0,0.647786,client_f369cb89fc
8521,106,122,0,0.626219,client_f369cb89fc
7026,106,307,1,0.626076,client_f369cb89fc
19278,106,357,0,0.626037,client_f369cb89fc
12232,106,388,1,0.626013,client_f369cb89fc
1732,106,461,1,0.625957,client_f369cb89fc
19045,106,741,0,0.625741,client_f369cb89fc
29102,106,752,1,0.625732,client_f369cb89fc
2574,106,1205,0,0.625382,client_f369cb89fc
27993,106,1266,0,0.625335,client_f369cb89fc


In [ ]:
false_positives = top_predictions[
    top_predictions["actual"] == 0
].head(5)

print("False positives among the highest-scored examples:")
display(false_positives)

print(
    "These examples show that a high model score does not guarantee "
    "that the evaluation label is 1."
)

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## Claim rewrite

### Earlier claim

> “The Week-5 logistic regression model predicts which content is declining and can be used to prioritize refresh opportunities.”

### Safer claim

> “In this experiment, a logistic-regression score using `days_since_last_update` and `impressions_90d` produced measured ranking performance on the held-out starter dataset. The grouped-by-client audit tests whether that performance remains directional on clients not seen during training. The score may be useful as decision support for prioritization, but the experiment does not establish causal impact or guarantee generalization to future clients.”

### Why I changed the claim

The original wording goes further than the validation evidence supports. The audit measures ranking performance, but it does not prove that using the model will improve content outcomes.

The revised wording therefore uses **observed, measured, directional, and decision-support** language and avoids claiming causation or guaranteed generalization.


## Self-check

* [x] I named two findings from the research paper.
* [x] I asked a constructive methodology question about the label/measurement for each finding.
* [x] I reproduced the Week-5 random split.
* [x] I evaluated the model using a grouped-by-client split.
* [x] I verified that there is zero client overlap between train and test.
* [x] I compared before/after Precision@10, Precision@20, and Precision@50.
* [x] I audited the feature list for target-derived and identifier leakage.
* [x] I inspected real failure examples from the grouped test set.
* [x] I rewrote my model claim using safe evidence-based language.
* [x] I ran the entire notebook from top to bottom.
* [x] I committed the executed notebook to GitHub.
